### Block 1: Setup, Config & Utils

In [1]:
!pip install jiwer transformers bitsandbytes scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.9 MB/s eta 0:00:00


In [2]:
# ============================================================================
# SETUP, IMPORTS & UTILS
# ============================================================================

import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import random
from scipy.ndimage import gaussian_filter1d

# Configuration
CONFIG = {
    'data_dir': '/kaggle/input/competitions/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'batch_size': 16,
    'num_epochs': 82,
    
    # Model architecture
    'd_model': 384,
    'n_heads': 6,
    'n_layers': 4,
    'd_ff': 1536,
    'patch_size': 3,
    'lstm_hidden': 256,
    'lstm_layers': 2,
    
    # Regularization & Adaptation
    'dropout': 0.4,
    'head_dim': 256,
    'attn_dropout': 0.5,
    'drop_path_rate': 0.2,       # NEW: Stochastic depth for Transformer
    'smooth_kernel_std': 2.0,    # NEW: Gaussian Smoothing standard deviation
    'smooth_kernel_size': 100,   # NEW: Gaussian Smoothing kernel size
    'drift_lambda': 0.01,        # NEW: Regularization for day-specific layers
    
    # Optimizer
    'learning_rate': 5e-4,
    'weight_decay': 1e-4,
    'use_augmentation': True,
}

print(f"Device: {CONFIG['device']}")
print(f"PyTorch version: {torch.__version__}")

# --- Helper: Stochastic Depth ---
def drop_path(x, drop_prob: float = 0., training: bool = False, scale_by_keep: bool = True):
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = x.new_empty(shape).bernoulli_(keep_prob)
    if keep_prob > 0.0 and scale_by_keep:
        random_tensor.div_(keep_prob)
    return x * random_tensor

# --- Helper: Global Session Mapper ---
def get_session2idx(data_dir):
    """Scans data directory to find all unique sessions chronologically."""
    from glob import glob
    paths = glob(f'{data_dir}/**/data_*.hdf5', recursive=True)
    sessions = sorted(list(set([Path(p).parent.name for p in paths])))
    return {s: i for i, s in enumerate(sessions)}

Device: cuda
PyTorch version: 2.10.0+cu128


### Block 2: Data Loading, Augmentation & Dataset

In [3]:
# ============================================================================
# DATA AUGMENTATION, LOADING & DATASET
# ============================================================================

class NeuralAugmentation:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, neural):
        if random.random() > self.p:
            return neural
        if random.random() < 0.3:
            neural = self.time_warp(neural)
        if random.random() < 0.3:
            noise_level = random.uniform(0.01, 0.05)
            neural = neural + torch.randn_like(neural) * noise_level
        if random.random() < 0.2:
            n_channels = neural.shape[1]
            n_drop = int(n_channels * 0.1)
            drop_indices = random.sample(range(n_channels), n_drop)
            neural[:, drop_indices] = 0
        return neural
    def time_warp(self, neural):
        seq_len = len(neural)
        warp_factor = random.uniform(0.9, 1.1)
        new_len = int(seq_len * warp_factor)
        if new_len < 10: return neural
        indices = torch.linspace(0, seq_len - 1, new_len)
        indices_floor = indices.long()
        indices_ceil = torch.clamp(indices_floor + 1, max=seq_len - 1)
        alpha = (indices - indices_floor.float()).unsqueeze(1)
        warped = (1 - alpha) * neural[indices_floor] + alpha * neural[indices_ceil]
        final_indices = torch.linspace(0, new_len - 1, seq_len).long()
        return warped[final_indices]

def load_split(data_dir, split='train'):
    from glob import glob
    pattern = f'{data_dir}/**/data_{split}.hdf5'
    files = sorted(glob(pattern, recursive=True))
    
    print(f"\nLoading {split} split...")
    all_data = {k: [] for k in ['neural', 'n_steps', 'sentence', 'phonemes', 
                                 'phoneme_len', 'session', 'block', 'trial']}
    for filepath in tqdm(files):
        session_name = Path(filepath).parent.name
        with h5py.File(filepath, 'r') as f:
            for trial_key in f.keys():
                trial = f[trial_key]
                all_data['neural'].append(trial['input_features'][:])
                all_data['n_steps'].append(trial.attrs['n_time_steps'])
                all_data['session'].append(session_name)
                all_data['block'].append(trial.attrs['block_num'])
                all_data['trial'].append(trial.attrs['trial_num'])
                
                sentence = trial.attrs.get('sentence_label')
                all_data['sentence'].append(sentence.decode('utf-8') if isinstance(sentence, bytes) else sentence)
                
                all_data['phonemes'].append(trial.get('seq_class_ids')[:] if 'seq_class_ids' in trial else None)
                all_data['phoneme_len'].append(trial.attrs.get('seq_len'))
    print(f"✓ Loaded {len(all_data['neural'])} samples")
    return all_data

class BrainToTextDataset(Dataset):
    def __init__(self, data, session2idx, char2idx=None, normalize=True, augment=False):
        self.neural = data['neural']
        self.n_steps = data['n_steps']
        self.sentences = data['sentence']
        self.sessions = data['session']
        self.session2idx = session2idx
        self.normalize = normalize
        self.augment = augment
        self.augmentation = NeuralAugmentation(p=0.5) if augment else None
        
        self.char2idx = char2idx if char2idx is not None else self._build_vocab()
        self.idx2char = {v: k for k, v in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)
    
    def _build_vocab(self):
        chars = set()
        for sent in self.sentences:
            if sent: chars.update(sent.lower())
        chars = sorted(list(chars))
        char2idx = {'<BLANK>': 0}
        for i, ch in enumerate(chars, start=1):
            char2idx[ch] = i
        return char2idx
    
    def __len__(self): return len(self.neural)
    
    def __getitem__(self, idx):
        neural = self.neural[idx][:self.n_steps[idx]]
        neural = torch.FloatTensor(neural)
        
        if self.normalize:
            neural = (neural - neural.mean()) / (neural.std() + 1e-8)
        if self.augment and self.augmentation:
            neural = self.augmentation(neural)
            
        sentence = self.sentences[idx] if self.sentences[idx] else ""
        target = [self.char2idx.get(ch.lower(), 0) for ch in sentence]
        
        return {
            'neural': neural,
            'target': torch.LongTensor(target),
            'length': len(neural),
            'target_length': len(target),
            'sentence': sentence,
            'day_idx': self.session2idx[self.sessions[idx]]  # NEW: Inject Session ID
        }

def collate_fn(batch):
    batch = sorted(batch, key=lambda x: x['length'], reverse=True)
    neurals = pad_sequence([item['neural'] for item in batch], batch_first=True)
    targets = pad_sequence([item['target'] for item in batch], batch_first=True)
    return {
        'neural': neurals,
        'target': targets,
        'lengths': torch.LongTensor([item['length'] for item in batch]),
        'target_lengths': torch.LongTensor([item['target_length'] for item in batch]),
        'sentences': [item['sentence'] for item in batch],
        'day_idx': torch.LongTensor([item['day_idx'] for item in batch])
    }

### Block 3: The Enriched Hybrid Model

In [4]:
# ============================================================================
# HYBRID MODEL (Enriched with Day-specific Weights & Gaussian Smooth)
# ============================================================================

class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048):
        super().__init__()
        half_dim = head_dim // 2
        freq = 1.0 / (10000 ** (torch.arange(0, half_dim, 2).float() / half_dim))
        t = torch.arange(max_seq_len).float().unsqueeze(1)
        angles = t * freq.unsqueeze(0)
        cos = torch.cos(angles).repeat_interleave(2, dim=1)
        sin = torch.sin(angles).repeat_interleave(2, dim=1)
        self.register_buffer("cos", cos.unsqueeze(0))
        self.register_buffer("sin", sin.unsqueeze(0))

    def forward(self, x, seq_len):
        cos = self.cos[:, :seq_len, :].to(x.device)
        sin = self.sin[:, :seq_len, :].to(x.device)
        x1, x2 = x.chunk(2, -1)
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1)

class WideAttention(nn.Module):
    def __init__(self, d_model, n_heads, head_dim, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.inner_dim = n_heads * head_dim
        self.scale = head_dim ** -0.5
        
        self.qkv = nn.Linear(d_model, self.inner_dim * 3, bias=False)
        self.out = nn.Linear(self.inner_dim, d_model)
        self.dropout = nn.Dropout(dropout)
        self.rope = RoPE(head_dim)
    
    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        q = self.rope(q, T)
        k = self.rope(k, T)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        
        attn = self.dropout(F.softmax(attn, -1))
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.out(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, head_dim, attn_dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = WideAttention(d_model, n_heads, head_dim, attn_dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x, mask=None, drop_path_rate=0.0):
        # Apply stochastic depth (drop path) around the residual connections
        x = x + drop_path(self.attn(self.norm1(x), mask), drop_path_rate, self.training)
        x = x + drop_path(self.ffn(self.norm2(x)), drop_path_rate, self.training)
        return x

class HybridLSTMTransformerCTC(nn.Module):
    def __init__(self, n_days, input_size=512, d_model=384, n_heads=6, n_layers=4,
                 d_ff=1536, patch_size=3, vocab_size=50, dropout=0.4, head_dim=256, 
                 attn_dropout=0.5, lstm_hidden=256, lstm_layers=2, smooth_std=2.0, 
                 smooth_size=100, drop_path_rate=0.2):
        super().__init__()
        self.patch_size = patch_size
        self.n_days = n_days
        
        # --- NEW: Gaussian Smoothing Initialization ---
        inp = np.zeros(smooth_size, dtype=np.float32)
        inp[smooth_size // 2] = 1
        gaussKernel = gaussian_filter1d(inp, smooth_std)
        valid_idx = np.argwhere(gaussKernel > 0.01)
        gaussKernel = gaussKernel[valid_idx]
        gaussKernel = np.squeeze(gaussKernel / np.sum(gaussKernel))
        self.register_buffer("gauss_kernel", torch.tensor(gaussKernel, dtype=torch.float32).view(1, 1, -1))
        
        # --- NEW: Day-Specific Linear Transformation ---
        self.day_weights = nn.ParameterList([nn.Parameter(torch.eye(input_size)) for _ in range(n_days)])
        self.day_biases = nn.ParameterList([nn.Parameter(torch.zeros(1, input_size)) for _ in range(n_days)])
        self.day_activation = nn.Softsign()

        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout * 0.5)
        )
        
        self.lstm = nn.LSTM(
            256, lstm_hidden, lstm_layers, batch_first=True, bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        lstm_output_dim = lstm_hidden * 2
        self.patch_embed = nn.Sequential(
            nn.LayerNorm(lstm_output_dim * patch_size),
            nn.Linear(lstm_output_dim * patch_size, d_model),
            nn.LayerNorm(d_model), nn.Dropout(dropout)
        )
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout, head_dim, attn_dropout)
            for _ in range(n_layers)
        ])
        
        self.drop_path_rates = [x.item() for x in torch.linspace(0, drop_path_rate, n_layers)]
        
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x, lengths, day_idx):
        B, T, C = x.shape
        
        # 1. Day Specific Weighting
        W = torch.stack([self.day_weights[i] for i in day_idx], dim=0)
        b = torch.cat([self.day_biases[i] for i in day_idx], dim=0).unsqueeze(1)
        x = torch.einsum("btd,bdk->btk", x, W) + b
        x = self.day_activation(x)

        # 2. Gaussian Smoothing via Conv1D
        x = x.permute(0, 2, 1) # [B, C, T]
        kernel = self.gauss_kernel.repeat(C, 1, 1).to(x.device)
        x = F.conv1d(x, kernel, padding='same', groups=C)
        
        # 3. CNN
        x = self.cnn(x)
        x = x.permute(0, 2, 1) # [B, T, C]
        
        # 4. LSTM
        x_packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        lstm_out, _ = self.lstm(x_packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        
        # 5. Patching
        T_lstm = lstm_out.shape[1]
        n_patches = T_lstm // self.patch_size
        
        if n_patches == 0:
            n_patches = 1
            x = lstm_out.mean(dim=1, keepdim=True)
            x = self.patch_embed(x.reshape(B, 1, -1))
            patch_lens = torch.ones(B, dtype=torch.long, device=x.device)
        else:
            x = lstm_out[:, :n_patches * self.patch_size].reshape(B, n_patches, -1)
            x = self.patch_embed(x)
            patch_lens = torch.clamp((lengths // self.patch_size).to(x.device), min=1)
        
        # 6. Transformer
        mask = (torch.arange(n_patches, device=x.device)[None, :] < patch_lens[:, None])
        mask = mask[:, None, None, :]
        
        for i, block in enumerate(self.blocks):
            x = block(x, mask, drop_path_rate=self.drop_path_rates[i])
        
        # 7. Output Projection
        logits = self.head(self.norm(x))
        log_probs = torch.log_softmax(logits, dim=-1)
        
        return log_probs.transpose(0, 1), patch_lens

### Block 4: Training & Validation Logic

# ============================================================================
# TRAINING (With Drift Loss)
# ============================================================================

import matplotlib.pyplot as plt
import jiwer
import torch

def validate_model(model, val_loader, idx2char, device):
    """
    Evaluates the model on the validation set and calculates the Word Error Rate (WER).
    """
    model.eval()
    predictions = []
    ground_truths = []
    
    with torch.no_grad():
        for batch in val_loader:
            neural = batch['neural'].to(device)
            lengths = batch['lengths']
            day_idx = batch['day_idx'].to(device)
            
            # Forward pass
            log_probs, output_lengths = model(neural, lengths, day_idx)
            
            # log_probs is shape [T, B, V], max_indices becomes [T, B]
            _, max_indices = log_probs.max(dim=-1)
            
            # CTC Decode each sequence in the batch
            for b in range(max_indices.size(1)):
                seq = max_indices[:output_lengths[b], b].cpu().numpy()
                decoded, prev = [], None
                for token in seq:
                    # Ignore blanks (0) and consecutive duplicates
                    if token != 0 and token != prev:
                        decoded.append(idx2char.get(token, ''))
                    prev = token
                
                predictions.append(''.join(decoded))
                ground_truths.append(batch['sentences'][b])
                
    # Filter strings to ensure jiwer does not crash on empty ground truths
    valid_preds, valid_truths = [], []
    for p, t in zip(predictions, ground_truths):
        if t and t.strip(): 
            # Provide a fallback string for empty predictions to avoid jiwer ValueError
            valid_preds.append(p if p.strip() else "<empty>") 
            valid_truths.append(t.strip())
            
    # Calculate Word Error Rate (WER)
    if not valid_truths:
        return 100.0  # Fallback if no valid targets exist
        
    wer = jiwer.wer(valid_truths, valid_preds) * 100.0
    return wer
    
def train_model(train_loader, val_loader, char2idx, config, n_days):
    model = HybridLSTMTransformerCTC(
        n_days=n_days,
        input_size=512, d_model=config['d_model'], n_heads=config['n_heads'],
        n_layers=config['n_layers'], d_ff=config['d_ff'], patch_size=config['patch_size'],
        vocab_size=len(char2idx), dropout=config['dropout'], head_dim=config['head_dim'],
        attn_dropout=config['attn_dropout'], lstm_hidden=config['lstm_hidden'],
        lstm_layers=config['lstm_layers'], smooth_std=config['smooth_kernel_std'],
        smooth_size=config['smooth_kernel_size'], drop_path_rate=config['drop_path_rate']
    ).to(config['device'])
    
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=config['learning_rate'], epochs=config['num_epochs'],
        steps_per_epoch=len(train_loader), pct_start=0.1
    )
    # ========================================================================
    # MINIMAL LOAD: Just load the model and optimizer before training starts
    # ========================================================================
    ckpt_path = '/kaggle/input/models/anibiswas1906125/result-v-01/pytorch/default/1/latest_model.pt'
    print(f"Loading model and optimizer from {ckpt_path}...")
    checkpoint = torch.load(ckpt_path, map_location=config['device'])
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print("Loaded successfully! Starting training...")
    # ========================================================================

    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=config['learning_rate'], epochs=config['num_epochs'],
        steps_per_epoch=len(train_loader), pct_start=0.1
    )
    # Track losses for plotting
    epoch_losses = []
    
    idx2char = {v: k for k, v in char2idx.items()}
    best_wer = float('inf')
    patience, no_improve = 10, 0
    
    for epoch in range(config['num_epochs']):
        model.train()
        train_loss = 0
        
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["num_epochs"]}'):
            neural = batch['neural'].to(config['device'])
            target = batch['target'].to(config['device'])
            lengths = batch['lengths']
            target_lengths = batch['target_lengths']
            day_idx = batch['day_idx'].to(config['device'])
            
            log_probs, output_lengths = model(neural, lengths, day_idx)
            ctc_loss = criterion(log_probs, target, output_lengths, target_lengths)
            
            drift_loss = 0.0
            if config['drift_lambda'] > 0 and n_days > 1:
                for d in range(1, n_days):
                    w_diff = model.day_weights[d] - model.day_weights[d-1]
                    b_diff = model.day_biases[d] - model.day_biases[d-1]
                    drift_loss += (torch.sum(w_diff ** 2) + torch.sum(b_diff ** 2))
                drift_loss = drift_loss / (n_days - 1)
                
            loss = ctc_loss + (config['drift_lambda'] * drift_loss)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
        
        avg_loss = train_loss / len(train_loader)
        epoch_losses.append(avg_loss)
        val_wer = validate_model(model, val_loader, idx2char, config['device'])
        
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, WER={val_wer:.2f}%")
        
        # Save Latest Checkpoint
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch
        }, 'latest_model.pt')

        # Save Best Checkpoint
        if val_wer < best_wer:
            best_wer = val_wer
            no_improve = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'char2idx': char2idx,
                'config': config,
                'wer': best_wer
            }, 'best_model.pt')
        else:
            no_improve += 1
            
        if no_improve >= patience:
            print(f"\nEarly stopping after {epoch+1} epochs")
            break
            
    # Plotting Loss Curve
    plt.figure(figsize=(10, 5))
    plt.plot(epoch_losses, label='Training Loss')
    plt.title('Training Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()
            
    return model, best_wer

### Block 5: Submission & Main Execution

# ============================================================================
# SUBMISSION CREATION & MAIN
# ============================================================================

def load_test_data_for_submission(data_dir, session2idx):
    from glob import glob
    print("\nLoading test data for submission...")
    pattern = f'{data_dir}/**/data_test.hdf5'
    files = sorted(glob(pattern, recursive=True))
    
    all_samples = []
    sample_id = 0
    for filepath in tqdm(files):
        session = Path(filepath).parent.name
        with h5py.File(filepath, 'r') as f:
            trial_keys = [k for k in f.keys() if 'trial' in k.lower()]
            for trial_key in trial_keys:
                trial = f[trial_key]
                if 'input_features' not in trial: continue
                
                features = trial['input_features'][:trial.attrs['n_time_steps']]
                features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)
                features = np.clip(features, -5, 5)
                
                all_samples.append({
                    'id': sample_id,
                    'session': session,
                    'day_idx': session2idx[session],
                    'trial_key': trial_key,
                    'features': torch.FloatTensor(features)
                })
                sample_id += 1
    return all_samples

def generate_predictions(model, test_samples, idx2char, device, batch_size=32):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(test_samples), batch_size)):
            batch = test_samples[i:i+batch_size]
            features = [s['features'] for s in batch]
            lengths = torch.LongTensor([len(f) for f in features])
            day_idx = torch.LongTensor([s['day_idx'] for s in batch]).to(device)
            
            features_padded = pad_sequence(features, batch_first=True).to(device)
            sorted_lengths, sorted_idx = lengths.sort(descending=True)
            
            features_sorted = features_padded[sorted_idx]
            day_idx_sorted = day_idx[sorted_idx]
            
            log_probs, output_lengths = model(features_sorted, sorted_lengths, day_idx_sorted)
            _, max_indices = log_probs.max(dim=-1)
            
            batch_preds = []
            for b in range(max_indices.size(1)):
                seq = max_indices[:output_lengths[b], b].cpu().numpy()
                decoded, prev = [], None
                for token in seq:
                    if token != 0 and token != prev:
                        decoded.append(idx2char.get(token, ''))
                    prev = token
                batch_preds.append(''.join(decoded))
            
            unsorted_preds = [''] * len(batch_preds)
            for j, pred in zip(sorted_idx.tolist(), batch_preds):
                unsorted_preds[j] = pred
            predictions.extend(unsorted_preds)
    return predictions

def main():
    print("="*80)
    print("ENHANCED HYBRID LSTM-TRANSFORMER PIPELINE")
    print("="*80)
    
    # Extract Global Session Indices
    session2idx = get_session2idx(CONFIG['data_dir'])
    n_days = len(session2idx)
    print(f"Discovered {n_days} unique sessions.")
    
    train_data = load_split(CONFIG['data_dir'], 'train')
    val_data = load_split(CONFIG['data_dir'], 'val')
    
    train_dataset = BrainToTextDataset(train_data, session2idx, augment=CONFIG['use_augmentation'])
    val_dataset = BrainToTextDataset(val_data, session2idx, char2idx=train_dataset.char2idx, augment=False)
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=2)
    
    model, best_wer = train_model(train_loader, val_loader, train_dataset.char2idx, CONFIG, n_days)
    
    test_samples = load_test_data_for_submission(CONFIG['data_dir'], session2idx)
    
    checkpoint = torch.load('best_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    idx2char = {v: k for k, v in checkpoint['char2idx'].items()}
    
    predictions = generate_predictions(model, test_samples, idx2char, CONFIG['device'], batch_size=32)
    
    df = pd.DataFrame({'id': [s['id'] for s in test_samples], 'text': predictions})
    df = df.sort_values('id').reset_index(drop=True)
    df['id'] = range(len(df))
    df.to_csv('submission.csv', index=False)
    print("\n✓ Pipeline complete. Expected target WER improved via Drift Loss and Gaussian Smoothing.")

if __name__ == "__main__":
    main()

### Force Reload Model & Load Datasets

In [5]:
# ============================================================================
# 1. FORCE RELOAD FROM BEST EPOCH & TEST DATA SETUP
# ============================================================================
import os
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import h5py
from glob import glob
from pathlib import Path
from tqdm import tqdm

print('Rebuilding session2idx / datasets / model from checkpoint...')
session2idx = get_session2idx(CONFIG['data_dir'])
n_days = len(session2idx)

train_data = load_split(CONFIG['data_dir'], 'train')
val_data = load_split(CONFIG['data_dir'], 'val')

train_dataset = BrainToTextDataset(train_data, session2idx, augment=False)
val_dataset = BrainToTextDataset(val_data, session2idx, char2idx=train_dataset.char2idx, augment=False)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=2)

CKPT_PATH = '/kaggle/input/models/anibiswas1906125/result-v-01/pytorch/default/1/best_model.pt' 
print(f"Loading checkpoint from: {CKPT_PATH}")
checkpoint = torch.load(CKPT_PATH, map_location=CONFIG['device'])
idx2char = {v: k for k, v in checkpoint['char2idx'].items()}
vocab_size = len(checkpoint['char2idx'])

model = HybridLSTMTransformerCTC(
    n_days=n_days, input_size=512, vocab_size=vocab_size,
    d_model=CONFIG['d_model'], n_heads=CONFIG['n_heads'], n_layers=CONFIG['n_layers'],
    d_ff=CONFIG['d_ff'], patch_size=CONFIG['patch_size'],
    lstm_hidden=CONFIG['lstm_hidden'], lstm_layers=CONFIG['lstm_layers'],
    dropout=CONFIG['dropout'], head_dim=CONFIG['head_dim'], attn_dropout=CONFIG['attn_dropout'],
    smooth_std=CONFIG['smooth_kernel_std'], smooth_size=CONFIG['smooth_kernel_size'],
    drop_path_rate=CONFIG['drop_path_rate'],
).to(CONFIG['device'])

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Localized Test Data Loader to bypass Block 5 completely
def block6_load_test_data(data_dir, session2idx):
    pattern = f'{data_dir}/**/data_test.hdf5'
    files = sorted(glob(pattern, recursive=True))
    all_samples = []
    sample_id = 0
    for filepath in tqdm(files, desc="Loading Test HDF5"):
        session = Path(filepath).parent.name
        with h5py.File(filepath, 'r') as f:
            trial_keys = [k for k in f.keys() if 'trial' in k.lower()]
            for trial_key in trial_keys:
                trial = f[trial_key]
                if 'input_features' not in trial: continue
                features = trial['input_features'][:trial.attrs['n_time_steps']]
                features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)
                features = np.clip(features, -5, 5)
                all_samples.append({
                    'id': sample_id, 'session': session, 'day_idx': session2idx[session],
                    'trial_key': trial_key, 'features': torch.FloatTensor(features)
                })
                sample_id += 1
    return all_samples

test_samples = block6_load_test_data(CONFIG['data_dir'], session2idx)
print(f"Setup Complete! Test samples: {len(test_samples)}")

Rebuilding session2idx / datasets / model from checkpoint...

Loading train split...


100%|██████████| 45/45 [03:17<00:00,  4.39s/it]


✓ Loaded 8072 samples

Loading val split...


100%|██████████| 41/41 [00:36<00:00,  1.11it/s]


✓ Loaded 1426 samples
Loading checkpoint from: /kaggle/input/models/anibiswas1906125/result-v-01/pytorch/default/1/best_model.pt


Loading Test HDF5: 100%|██████████| 41/41 [00:41<00:00,  1.02s/it]

Setup Complete! Test samples: 1450


In [6]:
# ============================================================================
# 2. INSTALL pyctcdecode + BUILD KenLM BINARIES (FIXED)
# ============================================================================
!pip install pyctcdecode kenlm -q

# 1. Update package lists first, then explicitly install the missing Boost packages
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq libboost-all-dev libboost-program-options-dev libboost-thread-dev libbz2-dev liblzma-dev > /dev/null 2>&1

# 2. Clean up any previous failed builds
!rm -rf kenlm 

# 3. Download and build
!wget -q -O - https://kheafield.com/code/kenlm.tar.gz | tar xz
!mkdir -p kenlm/build && cd kenlm/build && cmake .. > /dev/null && make -j4 > /dev/null

import os
KENLM_BIN = os.path.abspath('kenlm/build/bin')
assert os.path.exists(f'{KENLM_BIN}/lmplz'), 'lmplz build failed - Check CMake output'
print('KenLM binaries ready at', KENLM_BIN)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.5/427.5 kB 23.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 whic

In [7]:
# ============================================================================
# 3. BUILD TEXT CORPUS & TRAIN N-GRAM LM
# ============================================================================
corpus_path = 'lm_corpus.txt'
with open(corpus_path, 'w') as f:
    for sent in train_dataset.sentences:
        if sent: f.write(sent.lower().strip() + '\n')

arpa_path, arpa_fixed_path, binary_path = 'lm.arpa', 'lm_fixed.arpa', 'lm.binary'
!{KENLM_BIN}/lmplz -o 4 --discount_fallback < {corpus_path} > {arpa_path}

has_added_eos = False
with open(arpa_path, 'r') as r_file, open(arpa_fixed_path, 'w') as w_file:
    for line in r_file:
        if not has_added_eos and 'ngram 1=' in line:
            count = line.strip().split('=')[-1]
            w_file.write(line.replace(f'{count}', f'{int(count) + 1}'))
        elif not has_added_eos and '<s>' in line:
            w_file.write(line)
            w_file.write(line.replace('<s>', '</s>'))
            has_added_eos = True
        else:
            w_file.write(line)

!{KENLM_BIN}/build_binary {arpa_fixed_path} {binary_path}
print('KenLM binary status:', os.path.exists(binary_path))

=== 1/5 Counting and sorting n-grams ===
Reading /kaggle/working/lm_corpus.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 50649 types 5202
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:62424 2:4583393280 3:8593862656 4:13750180864
Statistics:
1 5202 D1=0.683574 D2=1.12152 D3+=1.50223
2 22757 D1=0.783016 D2=1.13272 D3+=1.54681
3 34722 D1=0.888569 D2=1.2482 D3+=1.36644
4 35492 D1=0.843176 D2=1.36002 D3+=1.86509
Memory estimate for binary LM:
type      kB
probing 2103 assuming -p 1.5
probing 2460 assuming -r models -p 1.5
trie     958 without quantization
trie     533 assuming -q 8 -b 8 quantization 
trie     896 assuming -a 22 array pointer compression
trie     472 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain s

In [8]:
# ============================================================================
# 4. MAIN DECODING EXECUTION WITH LLM RESCORING
# ============================================================================
from pyctcdecode import build_ctcdecoder
from transformers import AutoModelForCausalLM, AutoTokenizer
from multiprocessing import Pool
import torch.nn.functional as F

# 1. Initialize pyctcdecode
labels = ['' if idx2char[i] == '<BLANK>' else idx2char[i] for i in range(len(idx2char))]
decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=binary_path,
    alpha=0.4,   # Base acoustic weight
    beta=1.0     # Word insertion weight
)

# 2. Load Causal LLM for Perplexity Rescoring
LLM_NAME = "Qwen/Qwen2.5-7B"  # Replace with your path or choice of HuggingFace LLM
print(f"Loading Rescoring LLM: {LLM_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, 
    torch_dtype=torch.float16, 
    device_map="auto" # Auto allocates to your active GPU architecture
)
llm_model.eval()

def compute_llm_score(sentence):
    """Calculates pseudo-log-likelihood/loss for a string target using the LLM."""
    if not sentence.strip(): return -999.0
    inputs = tokenizer(sentence, return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        outputs = llm_model(**inputs, labels=inputs["input_ids"])
        # Cross entropy loss is negative log likelihood. Lower loss = more probable string
        return -outputs.loss.item() 

# 3. Generate emissions for Test Set
def extract_emissions(model, samples, device):
    model.eval()
    all_emissions, all_ids = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(samples), 32), desc='Extracting Test Emissions'):
            batch = samples[i:i + 32]
            features = [s['features'] for s in batch]
            lengths = torch.LongTensor([len(f) for f in features])
            day_idx = torch.LongTensor([s['day_idx'] for s in batch]).to(device)

            features_padded = torch.nn.utils.rnn.pad_sequence(features, batch_first=True).to(device)
            sorted_lengths, sorted_idx = lengths.sort(descending=True)

            log_probs, output_lengths = model(features_padded[sorted_idx], sorted_lengths, day_idx[sorted_idx])
            log_probs = log_probs.cpu().numpy()

            batch_emissions = [None] * len(batch)
            for b, orig_pos in enumerate(sorted_idx.tolist()):
                t_len = int(output_lengths[b])
                batch_emissions[orig_pos] = log_probs[:t_len, b, :]

            all_emissions.extend(batch_emissions)
            all_ids.extend([s['id'] for s in batch])
    return all_emissions, all_ids

test_emissions, test_ids = extract_emissions(model, test_samples, CONFIG['device'])

# ============================================================================
# 4. Perform Beam Search & Rescore Beams via LLM (FULLY CORRECTED)
# ============================================================================
FINAL_PREDICTIONS = []
BEAM_WIDTH = 100
LLM_WEIGHT = 1.0 # Tuning scalar adjusting the influence of the LLM vs CTC/KenLM

print("Executing Beam Search & LLM Rescoring...")
for emission in tqdm(test_emissions, desc="Rescoring Sequences"):
    # Generate Top K hypotheses paths using pyctcdecode's beam search
    beams = decoder.decode_beams(emission, beam_width=BEAM_WIDTH)
    
    best_text = ""
    best_combined_score = float('-inf')
    
    # Iterate through each candidate sentence in the beam
    for beam in beams:
        text = beam[0]  # The predicted string is always the first element in the tuple
        if not text: continue
        
        # Safely extract the AM and LM scores (they are the only float values in the tuple)
        scores = [item for item in beam if isinstance(item, float)]
        am_score = scores[0] if len(scores) > 0 else 0.0
        lm_score = scores[1] if len(scores) > 1 else 0.0
        
        # Calculate text fluency via the LLM
        llm_score = compute_llm_score(text)
        
        # Combine the acoustic score, n-gram score, and new LLM semantic score
        combined_score = am_score + lm_score + (LLM_WEIGHT * llm_score)
        
        if combined_score > best_combined_score:
            best_combined_score = combined_score
            best_text = text
            
    # Fallback to standard greedy decoding if beam yields empty string
    if not best_text:
        best_text = beams[0][0] if len(beams) > 0 else ""
        
    FINAL_PREDICTIONS.append(best_text)

# 5. Output Submission
df = pd.DataFrame({'id': test_ids, 'text': FINAL_PREDICTIONS})
df = df.sort_values('id').reset_index(drop=True)
df['id'] = range(len(df))
df.to_csv('submission_llm.csv', index=False)

print('\n✓ Finished! High-quality predictions written to submission_llm.csv')
display(df.head(10))

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
No known unigrams provided, decoding results might be a lot worse.


Loading Rescoring LLM: Qwen/Qwen2.5-7B...


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Extracting Test Emissions: 100%|██████████| 46/46 [00:26<00:00,  1.73it/s]


Executing Beam Search & LLM Rescoring...


Rescoring Sequences: 100%|██████████| 1450/1450 [43:40<00:00,  1.81s/it]


✓ Finished! High-quality predictions written to submission_llm.csv


,id,text
0,0,i get tired with the song and takes cortin.
1,1,imersaci gear.
2,2,you create a bige supprice.
3,3,i think famy you look at it.
4,4,show that they do have problems.
5,5,i enjoy that too.
6,6,i moved to gasas city.
7,7,maybe for times a week.
8,8,would he seems it to has effocente?
9,9,there's just no one around.
